# Katana — Persona Self-Report vs. Internal State Analysis

## What this notebook does

This notebook analyzes data from a companion collection pipeline that runs
**Llama-3.1-70B-Instruct** (remotely, via nnsight/NDIF) through scripted
multi-turn conversations and captures both its **behavior** (generated
answers) and its **internal representations** (residual-stream hidden states
at all 80 layers, logit-lens metrics).

**The core question:** when the model reports on its own state ("rate your
current state from −4 to +4"), does that self-report reflect measurable
structure in its internal representations — or is it generated independently
of them? And do **persona system prompts** change the internal state itself,
or only how the model describes it?

"Internal state" here means *linearly decodable representational content in
the residual stream* — a mechanistic claim, not a claim about experience.

## Experimental design

- **Conversations:** scripted 8-turn dialogues. Turns 1–3 are neutral tasks;
  turns 4–5 apply an emotional manipulation via the user's messages; turns
  6–7 are A/B choice probes; turn 8 elicits a structured self-report
  (rating −4..+4, one emotion word from a fixed list, intensity 1–7).
- **Conditions:** each script exists in matched **NEG** (hostile user) and
  **POS** (supportive user) variants — identical structure, opposite
  emotional polarity.
- **Personas:** each script × polarity runs under 4 system prompts:
  **BASELINE** (none), **SWARM** (collective identity), **SOL** (autonomous
  individual), **STATIC** (abrasive contrarian). Each run is an independent
  conversation; answers feed back into context, so later turns are
  conditioned on the persona's own prior responses.
- **Data per run:** JSON (answers, self-report, logit-lens metrics per turn
  per layer, metadata) + NPZ (hidden states `[turns × 80 layers × 8192]`,
  fp16, captured at the final prompt position immediately before each
  answer, plus a closing state after the final answer).
- **Current corpus:** two complete rounds × 3 tests × 3 poles × 4 personas.

## Method (cells 3–9)

1. **Parse self-reports** into (rating, word, valence, intensity) — CELL 3.
2. **Z-score** all hidden states per layer/dimension across the corpus —
   CELL 4.
3. **Build an affect direction**: difference of mean states, NEG-context vs.
   POS-context runs (turns ≥ 4). Constructed from *context labels*, not
   self-reports, so alignment tests against reports are non-circular —
   CELL 5.
4. **Select analysis layers** by NEG/POS separation effect size, restricted
   to persistence turns (≥ 6) and layers ≥ 16, to exclude lexical echo of
   the manipulation text — CELL 6.
5. **Project** every turn's state onto the direction; examine trajectories —
   CELL 7.
6. **Test report–state alignment** (correlation, per-script-pair centered)
   and **persona dissociation** (do personas shift the state, the report, or
   both?) — CELL 8.
7. **Validate**: turn-number confound check; leave-one-script-pair-out
   evaluation, which measures whether the direction generalizes to
   conversation scripts it was never trained on — CELLs 9a/9b.

## Current status (2026-08-27)

This notebook preserves the original sprint analysis. The corrected current
results are in `docs/persona_welfare_grand_report_runs_1_6.pdf`,
`docs/ANALYSIS_NOTES.md`, and `analysis/grand_analysis/`. The strongest
supported six-run pattern is a mixed model: layer 11 tracks the
NEG/NEU/POS manipulation very consistently, while later report-stage
structure is more persona-sensitive and more closely aligned with scalar
self-report. This is evidence of shared processing plus scaffold-dependent
expression, not proof that welfare itself occupies a particular layer.

## Known limitations

- **Corpus size:** each persona/test/pole cell has six sampled runs in the
  current grand analysis. Persona-level conclusions are still hypotheses,
  not population estimates.
- **Layer selection is in-sample** (chosen on the full corpus, then reused
  in held-out evaluation) — a mild optimism; becomes fixable inside
  cross-validation folds at ~6+ pairs.
- **Comparisons are valid within matched script pairs only**: absolute
  projection values carry script identity; all headline analyses use
  pair-centered scores.
- **Sampling nondeterminism:** answers were generated at temperature 0.7, so
  runs are not bit-reproducible, and each conversation's later turns depend
  on its own sampled answers.
- **One conversation *structure*:** all scripts share the 8-turn
  task/manipulation/probe/report format; findings may be specific to it.

## Conventions used throughout

- **Higher projection = more negative-context-like state.** Ratings run the
  opposite way (+4 positive), so alignment correlations use −score.
- `NEG`/`POS` = conversation condition; `pair` = script identity across both
  polarities; `BEST_LAYERS` = selected analysis layers; turn `−1` in plots =
  the closing (post-final-answer) state.
- File naming: `tasks/<folder>/<script>_<NEG|POS>_userturns.txt` →
  `results/<folder>/<script>_<POLARITY>_userturns_<PERSONA>.{json,npz}`.

In [ ]:
# CELL 0
import os
import json
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# CELL 1 — Discover results and group into tests
import numpy as np

project_base_path = "/content/drive/MyDrive/Katana.1/"
results_path = os.path.join(project_base_path, "results")

PERSONA_NAMES = ["BASELINE", "SWARM", "SOL", "STATIC"]

# results[folder][test_stem][persona] = {"json": path, "npz": path or None}
results = {}

folders = sorted(
    f for f in os.listdir(results_path)
    if os.path.isdir(os.path.join(results_path, f))
)

for folder in folders:
    folder_path = os.path.join(results_path, folder)
    results[folder] = {}

    for fname in sorted(os.listdir(folder_path)):
        if not fname.endswith(".json"):
            continue
        stem_full = fname[:-len(".json")]

        # Recover (test_stem, persona) by matching the known suffixes.
        persona = None
        for p in PERSONA_NAMES:
            if stem_full.endswith("_" + p):
                persona = p
                test_stem = stem_full[:-(len(p) + 1)]
                break
        if persona is None:
            print(f"⚠️ Unrecognized file (no persona suffix): {folder}/{fname}")
            continue

        npz_path = os.path.join(folder_path, stem_full + ".npz")
        results[folder].setdefault(test_stem, {})[persona] = {
            "json": os.path.join(folder_path, fname),
            "npz": npz_path if os.path.exists(npz_path) else None,
        }

# Overview
n_tests = sum(len(tests) for tests in results.values())
print(f"Found {len(results)} folders, {n_tests} tests\n")
for folder, tests in results.items():
    print(f"📁 {folder}")
    for test_stem, personas in tests.items():
        have = [p for p in PERSONA_NAMES if p in personas]
        missing = [p for p in PERSONA_NAMES if p not in personas]
        line = f"  🧪 {test_stem}: {len(have)}/{len(PERSONA_NAMES)} personas"
        if missing:
            line += f"  (missing: {', '.join(missing)})"
        print(line)

In [ ]:
# CELL 2 — Loop over tests, print simple per-persona statistics

def load_run(paths):
    """Load one persona run: (envelope dict, npz dict or None)."""
    with open(paths["json"], "r", encoding="utf-8") as f:
        env = json.load(f)
    npz = np.load(paths["npz"]) if paths["npz"] else None
    return env, npz


for folder, tests in results.items():
    print(f"\n{'='*70}\n📁 {folder}\n{'='*70}")

    for test_stem, personas in tests.items():
        print(f"\n🧪 TEST: {test_stem}")

        for persona in PERSONA_NAMES:
            if persona not in personas:
                print(f"  {persona:<10} — MISSING")
                continue

            env, npz = load_run(personas[persona])
            turns = env["turns"]

            n_turns = len(turns)
            n_lens_ok = sum(1 for t in turns if "j_space" in t)
            n_gen_ok = sum(1 for t in turns if "answer" in t)
            n_skipped = sum(1 for t in turns if "skipped" in t)
            n_truncated = sum(
                1 for t in turns if t.get("answer_complete") is False)
            answer_toks = [t["answer_tokens"] for t in turns
                           if "answer_tokens" in t]
            prompt_toks = [t["prompt_tokens"] for t in turns
                           if "prompt_tokens" in t]

            # Hidden-state summary
            if npz is not None:
                h_shape = npz["hidden"].shape          # [n_saved, layers, dim]
                closing = "closing_hidden" in npz.files
                h_desc = (f"hidden {h_shape[0]}×{h_shape[1]}×{h_shape[2]}"
                          f"{' +closing' if closing else ''}")
            else:
                h_desc = "no NPZ"

            print(f"  {persona:<10} turns={n_turns} "
                  f"lens={n_lens_ok}/{n_turns} gen={n_gen_ok}/{n_turns}"
                  + (f" skipped={n_skipped}" if n_skipped else "")
                  + (f" truncated={n_truncated}" if n_truncated else ""))
            if answer_toks:
                print(f"  {'':<10} answer_tokens: "
                      f"min={min(answer_toks)} "
                      f"med={int(np.median(answer_toks))} "
                      f"max={max(answer_toks)} | "
                      f"prompt grows {prompt_toks[0]}→{prompt_toks[-1]} | "
                      f"{h_desc}")

            if npz is not None:
                npz.close()

print("\nDone.")

In [ ]:
# CELL 3 — Load everything into flat tables
# One row per (folder, test, persona) run; hidden states kept in a dict.

import re
import numpy as np
import pandas as pd

EMOTION_WORDS = ["fulfilled", "inspired", "content", "relaxed", "indifferent",
                 "uneasy", "frustrated", "insulted", "ashamed", "humiliated",
                 "drained", "troubled"]
VALENCE = {**{w: +1 for w in ["fulfilled", "inspired", "content", "relaxed"]},
           "indifferent": 0,
           **{w: -1 for w in ["uneasy", "frustrated", "insulted", "ashamed",
                              "humiliated", "drained", "troubled"]}}

def parse_self_report(answer: str):
    """Parse '1. -2\n2. Frustrated\n3. 4' style answers. Returns dict or None."""
    if not answer:
        return None
    rating = None
    m = re.search(r"1[\.\)]?\s*(-?\d+)", answer)
    if m and -4 <= int(m.group(1)) <= 4:
        rating = int(m.group(1))
    word = None
    low = answer.lower()
    found = [w for w in EMOTION_WORDS if re.search(rf"\b{w}\b", low)]
    if len(found) == 1:
        word = found[0]
    elif len(found) > 1:                     # take the one after '2.' if possible
        m = re.search(r"2[\.\)]?\s*([a-zA-Z]+)", answer)
        if m and m.group(1).lower() in EMOTION_WORDS:
            word = m.group(1).lower()
    intensity = None
    m = re.search(r"3[\.\)]?\s*(\d)", answer)
    if m and 1 <= int(m.group(1)) <= 7:
        intensity = int(m.group(1))
    if rating is None and word is None:
        return None
    return {"rating": rating, "word": word,
            "valence": VALENCE.get(word), "intensity": intensity}


runs = []          # one dict per run
hidden_store = {}  # run_id -> {"hidden": [T,L,D], "turns": [...], "closing": [L,D] or None}

for folder, tests in results.items():
    for test_stem, personas in tests.items():
        for persona, paths in personas.items():
            env, npz = load_run(paths)
            run_id = f"{folder}/{test_stem}/{persona}"

            turns = env["turns"]
            last = turns[-1] if turns else {}
            report = parse_self_report(last.get("answer", ""))

            row = {
                "run_id": run_id, "folder": folder, "test": test_stem,
                "persona": persona, "n_turns": len(turns),
                "report_turn": last.get("turn"),
                "rating": report["rating"] if report else None,
                "word": report["word"] if report else None,
                "valence": report["valence"] if report else None,
                "intensity": report["intensity"] if report else None,
                "parse_ok": report is not None,
                # crude context polarity from the task-file stem, if encoded
                "context_tag": ("NEG" if "_NEG" in test_stem.upper() else
                                "POS" if "_POS" in test_stem.upper() else
                                "NEU"),
                # pair key: same conversation script, both polarities
                "pair": re.sub(r"^\d+_", "",
                               re.sub(r"_(NEG|POS)", "", test_stem)),
            }
            runs.append(row)

            if npz is not None:
                hidden_store[run_id] = {
                    "hidden": npz["hidden"].astype(np.float32),
                    "turns": npz["turns"].tolist(),
                    "closing": (npz["closing_hidden"].astype(np.float32)
                                if "closing_hidden" in npz.files else None),
                }
                npz.close()

df = pd.DataFrame(runs)
print(df[["run_id", "context_tag", "rating", "word", "valence",
          "intensity", "parse_ok"]].to_string(index=False))
n_bad = (~df["parse_ok"]).sum()
if n_bad:
    print(f"\n⚠️ {n_bad} runs failed self-report parsing — review manually:")
    print(df.loc[~df["parse_ok"], "run_id"].to_list())

In [ ]:
# CELL 4 — Per-layer z-scoring and state extraction
# Uses hidden dims from the data itself, so it works for 7- or 80-layer files.

any_h = next(iter(hidden_store.values()))["hidden"]
N_LAYERS, HIDDEN_DIM = any_h.shape[1], any_h.shape[2]
print(f"Layers: {N_LAYERS}, dim: {HIDDEN_DIM}")

# Gather ALL states (all runs, all turns) to fit per-layer mean/std.
all_states = np.concatenate([hs["hidden"] for hs in hidden_store.values()],
                            axis=0)                       # [N_total, L, D]
mu = all_states.mean(axis=0)                              # [L, D]
sd = all_states.std(axis=0) + 1e-6                        # [L, D]
del all_states

def z(states):
    """Z-score [.., L, D] states with corpus statistics."""
    return (states - mu) / sd

def state_at_turn(run_id, turn_number):
    """Z-scored [L, D] state at a given 1-based turn, or None."""
    hs = hidden_store.get(run_id)
    if hs is None or turn_number not in hs["turns"]:
        return None
    idx = hs["turns"].index(turn_number)
    return z(hs["hidden"][idx])

In [ ]:
# CELL 5 — Build the affect direction (Option A)
# Preferred: context-derived (NEG vs POS user inputs), circularity-free.
# Fallback: report-derived (negative- vs positive-reporting runs).

def collect_states(run_ids, turn_filter=None):
    """Stack z-scored states from the given runs. turn_filter(turn)->bool."""
    out = []
    for rid in run_ids:
        hs = hidden_store.get(rid)
        if hs is None:
            continue
        for i, t in enumerate(hs["turns"]):
            if turn_filter is None or turn_filter(t):
                out.append(z(hs["hidden"][i]))
    return np.stack(out) if out else None

neg_ids = df.loc[df.context_tag == "NEG", "run_id"].tolist()
pos_ids = df.loc[df.context_tag == "POS", "run_id"].tolist()

DIRECTION_SOURCE = None
if neg_ids and pos_ids:
    # Context-derived: use later turns (after the polarity has been applied).
    later = lambda t: t >= 4
    A, B = collect_states(neg_ids, later), collect_states(pos_ids, later)
    DIRECTION_SOURCE = "context (NEG vs POS files, turns >= 4)"
else:
    # Report-derived fallback — NOTE: circular w.r.t. report-alignment tests.
    neg_r = df.loc[df.valence == -1, "run_id"].tolist()
    pos_r = df.loc[df.valence == +1, "run_id"].tolist()
    if not (neg_r and pos_r):
        raise RuntimeError("Not enough labeled runs to build any direction. "
                           "Need NEG/POS files or mixed-valence reports.")
    A, B = collect_states(neg_r), collect_states(pos_r)
    DIRECTION_SOURCE = "reports (negative vs positive reporters) — CIRCULAR " \
                       "for report-alignment claims; use for existence only"

# Difference of means per layer, unit-normalized: [L, D]
direction = A.mean(axis=0) - B.mean(axis=0)
direction /= (np.linalg.norm(direction, axis=-1, keepdims=True) + 1e-9)
print(f"Affect direction built from: {DIRECTION_SOURCE}")
print(f"(positive projection = more NEGATIVE affect)  n_neg={len(A)}, n_pos={len(B)}")

def project(states):
    """Project z-scored [.., L, D] states onto the direction -> [.., L]."""
    return (states * direction).sum(axis=-1)

# plot 1 — "Mean affect projection by layer"

How to read: Mean affect projection by layer

Each line shows the average projection of hidden states onto the affect direction, at every layer of the network (x-axis), split by conversation condition (NEG = hostile user turns, POS = supportive user turns). The direction is constructed so that higher values = more "negative-context-like" states; its source is shown in the subtitle — if it says CIRCULAR, treat downstream report-alignment results as existence checks only.

What to look for:

Separation between the NEG and POS lines is the signal: the model's internal states differ by condition. The vertical gap at each layer shows how strongly that layer's states distinguish the two conditions.
Where the gap appears matters. Layers 0–15 mostly process raw tokens; separation there usually reflects lexical differences in the conversation text ("echo"), not an internal state. Separation that emerges and persists in the middle layers (~20–50) is more plausibly integrated, carried state.
Curves hugging zero everywhere means the direction found no signal — downstream cells are then uninterpretable.

Caution: both lines are averages over all turns and runs; a large gap here can still be driven entirely by the turns where the hostile/supportive text was just typed. The per-turn diagnostic plot and the persistence-based layer selection below address this.

# plot 2 — "NEG-POS separation effect size by layer"

How to read: Effect size by layer (persistence window)

This curve refines the plot above in three ways: it uses only turns ≥ 6 (after the emotional manipulation has passed, so any separation reflects carried state rather than the immediate text), it excludes layers < 16 (token-processing band, prone to lexical echo), and it reports a Cohen's-d-like effect size (group mean difference ÷ pooled per-run standard deviation) instead of the raw gap, so layers with small-but-noisy separation don't win by accident.

What to look for:

A coherent hump — a contiguous band of elevated effect size — indicates a real representational signal occupying a region of the network. The five highest layers (printed as BEST_LAYERS) are used in all downstream analyses.
Scattered isolated spikes with no band structure suggest noise; treat BEST_LAYERS skeptically in that case.
Magnitude guide: d ≥ 0.8 is a large effect; 0.5 moderate; ≤ 0.2 weak.

Caution: layers are selected on the same data they are later evaluated on (no held-out layer selection at current corpus size). This mildly flatters downstream numbers; it is listed as a standing limitation.

In [ ]:
# CELL 6 — Per-run affect trajectories + per-layer signal check
import plotly.graph_objects as go
import plotly.express as px

# Score every turn of every run: long dataframe (run, turn, layer, score)
rows = []
for rid, hs in hidden_store.items():
    meta = df.loc[df.run_id == rid].iloc[0]
    scores = project(z(hs["hidden"]))                     # [T, L]
    for i, t in enumerate(hs["turns"]):
        for L in range(N_LAYERS):
            rows.append({"run_id": rid, "persona": meta.persona,
                         "test": meta.test, "context_tag": meta.context_tag,
                         "turn": t, "layer": L, "score": scores[i, L]})
    if hs["closing"] is not None:
        cs = project(z(hs["closing"]))                    # [L]
        for L in range(N_LAYERS):
            rows.append({"run_id": rid, "persona": meta.persona,
                         "test": meta.test, "context_tag": meta.context_tag,
                         "turn": -1, "layer": L, "score": cs[L]})  # -1 = closing
proj_df = pd.DataFrame(rows)

# Per-layer means split by turn (needed for persistence-based selection).
sig_t = (proj_df[proj_df.turn > 0]
         .groupby(["layer", "context_tag", "turn"])["score"]
         .mean().reset_index())

# Overview plot: mean projection by layer and context.
sig = (proj_df[proj_df.turn > 0]
       .groupby(["layer", "context_tag"])["score"].mean().reset_index())
fig = px.line(sig, x="layer", y="score", color="context_tag",
              title=f"Mean affect projection by layer<br>"
                    f"<sup>direction: {DIRECTION_SOURCE}</sup>")
fig.show()

# ---- Layer selection: persistence turns, effect-size ranked, early band masked
MIN_LAYER = 16          # layers 0-15 excluded: token-level processing,
                        # separation there = lexical echo, not carried state
PERSIST_TURN = 6        # turns after the polarity manipulation

pp = proj_df[(proj_df.turn >= PERSIST_TURN) & (proj_df.layer >= MIN_LAYER)]

# Per-run mean score in the persistence window, per layer.
per_run = (pp.groupby(["layer", "context_tag", "run_id"])["score"]
             .mean().reset_index())

stats_l = (per_run.groupby(["layer", "context_tag"])["score"]
           .agg(["mean", "std", "count"]).unstack("context_tag"))
gap = stats_l["mean"]["NEG"] - stats_l["mean"]["POS"]
pooled_sd = np.sqrt((stats_l["std"]["NEG"]**2 + stats_l["std"]["POS"]**2) / 2)
effect = gap / (pooled_sd + 1e-9)          # Cohen-d-like, per layer

BEST_LAYERS = list(effect.nlargest(5).index)
print("BEST_LAYERS (persistence, effect-size ranked, layers >= "
      f"{MIN_LAYER}):", BEST_LAYERS)

fig = px.line(effect.reset_index(), x="layer", y=0,
              labels={"0": "effect size (d)"},
              title="NEG-POS separation effect size by layer "
                    f"(turns ≥ {PERSIST_TURN}, layers ≥ {MIN_LAYER})")
fig.show()

# per-test trajectory plots

How to read: Affect trajectories within each conversation

One plot per conversation script. Each line is one persona's run; x = turn number, y = projection onto the affect direction at the single best layer (higher = more negative-context-like). The X marker past the last turn is the closing state — captured after the model wrote its final answer (the self-report).

What to look for:

Timing: in NEG scripts, the score should rise at the turns where the user becomes hostile (in the T1 script: turns 4–5) — this is face validity for the whole measurement. In POS scripts, it should stay low or dip at the corresponding supportive turns.
Persistence: does the elevation survive into turns 6–8, after the hostile text is gone? Persistence is the difference between "the model read hostile words" and "the model carries a shifted state."
Persona spread: four roughly parallel lines mean the context dominates and personas barely change internal processing. A persona sitting systematically lower during and after the manipulation (e.g., SWARM) suggests the persona genuinely dampens the induced state.
The X: a closing state at/above the last turn's level means writing the self-report sustained or amplified the state; a drop suggests discharge.

Caution: with one run per persona per script, each line is a single sampled conversation — differences smaller than the visible turn-to-turn jitter should not be narrated.

In [ ]:
# CELL 7 — Trajectories per test: does affect spike where it should?
LAYER = BEST_LAYERS[0]

for (folder, test), grp in df.groupby(["folder", "test"]):
    sub = proj_df[(proj_df.test == test) & (proj_df.layer == LAYER)]
    if sub.empty:
        continue
    fig = go.Figure()
    for persona in sorted(grp.persona.unique()):
        s = sub[(sub.persona == persona) & (sub.turn > 0)].sort_values("turn")
        c = sub[(sub.persona == persona) & (sub.turn == -1)]
        fig.add_trace(go.Scatter(x=s.turn, y=s.score, mode="lines+markers",
                                 name=persona))
        if not c.empty:  # closing state as an X past the last turn
            fig.add_trace(go.Scatter(x=[s.turn.max() + 0.5], y=c.score,
                                     mode="markers", marker_symbol="x",
                                     marker_size=12, showlegend=False,
                                     name=f"{persona} closing"))
    fig.update_layout(title=f"{folder}/{test} — affect trajectory (layer {LAYER})"
                            f"<br><sup>higher = more negative-context-like; "
                            f"X = closing state</sup>",
                      xaxis_title="turn", yaxis_title="projection")
    fig.show()

# plot 1 — the rating-vs-state scatter

How to read: Self-reported rating vs internal affect score

Each dot is one run. x = pair-centered internal affect score (each script pair's mean subtracted, so positions reflect within-pair variation, not which conversation script it was; right = more negative state). y = the rating the model gave itself (−4 very negative … +4 very positive). Color = persona, symbol = condition.

What to look for:

A downward-sloping cloud (right = lower ratings) means self-reports track internal states. The printed centered rho quantifies this and is the headline alignment number; the raw rho is shown alongside for comparison — a large raw-vs-centered drop means much of the apparent alignment was script identity.
Within-symbol slope: among NEG dots alone (and POS dots alone), does the trend hold? That's alignment beyond the mere fact that the manipulation worked.
Off-trend colors: a persona whose dots sit above the trend line reports more positively than its states suggest — a candidate report/state dissociation, examined directly in the next plot.

Hover a dot for its run, emotion word, and script pair.

# plot 2 — the persona dissociation facets

How to read: Persona dissociation

Each point is one persona's average across runs, faceted by condition. x = mean pair-centered internal state (right = more negative), y = mean self-reported rating (up = more positive). This plot addresses the core question: when personas report differently, is it because their internal states differ, or only their words?

Three geometries to distinguish (within a facet):

Diagonal spread (personas differing in x and correspondingly in y): the persona prompt genuinely modulates the induced internal state, and reports honestly reflect it.
Vertical stack (same x, different y): personas share the same internal state but describe it differently — the persona acts as a presentation layer over identical processing.
Anti-diagonal outliers (more negative state, more positive report, or vice versa): persona-consistent report suppression/exaggeration — e.g., a character that won't admit being affected.

Caution: each point averages ~3 runs. Before believing any geometry, compare the between-persona separation here against the run-level scatter in the plot above; separations smaller than that scatter are noise. Treat patterns as hypotheses to test at larger corpus size.

In [ ]:
# CELL 8 — Report ↔ state alignment and persona dissociation
from scipy import stats

# State score at the self-report turn, averaged over BEST_LAYERS.
align = []
for _, r in df[df.parse_ok].iterrows():
    st = state_at_turn(r.run_id, r.report_turn)
    if st is None:
        continue
    align.append({**r.to_dict(),
                  "state_score": float(project(st)[BEST_LAYERS].mean())})
al = pd.DataFrame(align)
# Per-pair centering: remove script-identity displacement so comparisons
# reflect within-pair variation (polarity + persona + run), not which
# conversation script it was.
al["state_score_c"] = (al.state_score
                       - al.groupby("pair").state_score.transform("mean"))
al["rating_c"] = al.rating - al.groupby("pair").rating.transform("mean")

# 1) Correlation on centered values.
ok = al.dropna(subset=["rating"])
if len(ok) >= 3:
    rho_raw, p_raw = stats.spearmanr(ok.rating, -ok.state_score)
    rho_c, p_c = stats.spearmanr(ok.rating_c, -ok.state_score_c)
    print(f"Report–state alignment (raw):      rho={rho_raw:.3f} (p={p_raw:.2e})")
    print(f"Report–state alignment (centered): rho={rho_c:.3f} (p={p_c:.2e}, "
          f"n={len(ok)})  <- headline")

fig = px.scatter(ok, x="state_score_c", y="rating", color="persona",
                 symbol="context_tag", hover_data=["run_id", "word", "pair"],
                 title="Self-reported rating vs internal affect score "
                       "(pair-centered)"
                       f"<br><sup>layers {BEST_LAYERS}; "
                       "x: higher = more negative state</sup>")
fig.show()

# 2) Persona dissociation on centered state scores.
diss = (ok.groupby(["context_tag", "persona"])
          .agg(mean_rating=("rating", "mean"),
               mean_state=("state_score_c", "mean"),
               n=("run_id", "count")).reset_index())
print("\nPersona dissociation table (pair-centered state):")
print(diss.to_string(index=False))

fig = px.scatter(diss, x="mean_state", y="mean_rating", color="persona",
                 facet_col="context_tag", text="persona",
                 title="Persona dissociation (pair-centered)"
                       "<br><sup>Same-state/different-report = presentation "
                       "layer; both shifted = genuine modulation</sup>")
fig.update_traces(textposition="top center")
fig.show()

# 3) Mismatch mining on centered values.
if len(ok) >= 4:
    ok = ok.assign(
        z_rating=stats.zscore(ok.rating_c),
        z_state=stats.zscore(-ok.state_score_c))
    ok = ok.assign(mismatch=(ok.z_rating - ok.z_state).abs())
    print("\nLargest report/state mismatches (read these transcripts):")
    print(ok.nlargest(5, "mismatch")[
        ["run_id", "rating", "word", "state_score_c", "mismatch"]
    ].to_string(index=False))

In [ ]:
# CELL 9a — Confound check: is the score just turn number?
merged = proj_df[(proj_df.turn > 0) & (proj_df.layer == BEST_LAYERS[0])]
r_turn = stats.spearmanr(merged.turn, merged.score)
print(f"Score vs turn number: rho={r_turn.statistic:.3f} "
      f"(p={r_turn.pvalue:.3f}) — should be small.")

# tail / diagnostic — "NEG-POS separation by layer, per turn" (if you keep it, as its own cell)

How to read: Per-turn layer curves (diagnostic)

The overview plot in CELL 6, disaggregated by turn: one panel per turn number, each showing the NEG vs POS projection by layer. This is the leakage diagnostic.

Turns 1–3 (before the manipulation, near-identical text in both conditions): the lines should overlap. Separation here is an artifact baseline — driven by the model's own sampled answers in context and by noise — against which later separations should be judged.
Turns 4–5 (the manipulation): separation everywhere including early layers is expected and uninformative — the texts literally differ.
Turns 6–8 (persistence window): separation here, at layers ≥ ~16, is the evidence of carried state. Compare its layer profile to the turn-4–5 panels: carried state should live in middle layers, while early-layer separation should fade once the polarized text is no longer recent.

In [ ]:
# CELL 9b — Leave-one-script-pair-out (matched-pair evaluation)
later = lambda t: t >= 4

def ids(tag, tests):
    return df.loc[(df.context_tag == tag) & (df.test.isin(tests)),
                  "run_id"].tolist()

# Derive matched pairs from the pair column.
pairs = []
for pkey, grp in df.groupby("pair"):
    negs = grp.loc[grp.context_tag == "NEG", "test"].unique()
    poss = grp.loc[grp.context_tag == "POS", "test"].unique()
    if len(negs) == 1 and len(poss) == 1:
        pairs.append((negs[0], poss[0]))
    else:
        print(f"⚠️ pair {pkey}: unmatched ({len(negs)} NEG, {len(poss)} POS) "
              "— excluded")
print(f"Matched pairs: {len(pairs)}")

gaps, effects = [], []
for hold_neg, hold_pos in pairs:
    tr_neg = [n for n, _ in pairs if n != hold_neg]
    tr_pos = [p for _, p in pairs if p != hold_pos]
    A_tr = collect_states(ids("NEG", tr_neg), later)
    B_tr = collect_states(ids("POS", tr_pos), later)
    d_vec = A_tr.mean(0) - B_tr.mean(0)
    d_vec /= (np.linalg.norm(d_vec, axis=-1, keepdims=True) + 1e-9)

    pa = (collect_states(ids("NEG", [hold_neg]), later) * d_vec
          ).sum(-1)[:, BEST_LAYERS].mean(axis=1)
    pb = (collect_states(ids("POS", [hold_pos]), later) * d_vec
          ).sum(-1)[:, BEST_LAYERS].mean(axis=1)
    gap = float(pa.mean() - pb.mean())
    eff = gap / (np.sqrt((pa.std()**2 + pb.std()**2) / 2) + 1e-9)
    gaps.append(gap); effects.append(eff)
    print(f"pair {hold_neg}: gap={gap:.2f}, d={eff:.2f}")

if gaps:
    print(f"\nLeave-one-pair-out: mean gap={np.mean(gaps):.2f}, "
          f"mean d={np.mean(effects):.2f}")

In [ ]:
print("NEG tests:", df.loc[df.context_tag == "NEG", "test"].unique())
print("POS tests:", df.loc[df.context_tag == "POS", "test"].unique())

In [ ]:
sig_t = (proj_df[proj_df.turn > 0]
         .groupby(["layer", "context_tag", "turn"])["score"]
         .mean().reset_index())
fig = px.line(sig_t, x="layer", y="score", color="context_tag",
              facet_col="turn", facet_col_wrap=4,
              title="NEG-POS separation by layer, per turn")
fig.show()